# Notebook 3: A multi-tool agent

Now the agent has **three** tools: a calculator, a fact lookup, and a greeting
tool. The model chooses the right one for each request by reading the tool
descriptions. We reuse the ReAct loop from Notebook 2.

## Setup: run these first

Every notebook starts the same way: install the library, load your key, and
pick a model. Run the next three cells in order.

**Save yourself time:** store your API key once in Colab Secrets (the key icon on
the left sidebar). Add a secret named `ANTHROPIC_API_KEY`, paste your key as the
value, and turn on "Notebook access". After that, every notebook picks it up
automatically and you never paste it again.

In [ ]:
# 1) Install the Anthropic library (Colab forgets it between sessions, so we
#    install each time). The -q just keeps the output quiet.
!pip install -q anthropic

In [ ]:
# 2) Load your Anthropic API key.
#    Easiest: store it once in Colab Secrets (key icon on the left), named
#    ANTHROPIC_API_KEY, with "Notebook access" on. Then this reads it for you.
import anthropic

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = None  # not on Colab, or no secret set yet

if not api_key:
    # Fallback: ask once if no secret is found. The input is hidden and not saved.
    from getpass import getpass
    api_key = getpass("Paste your Anthropic API key (input is hidden): ")

client = anthropic.Anthropic(api_key=api_key)
print("Your client is ready.")

In [ ]:
# 3) Choose the model. claude-opus-4-8 is the most capable. For faster, cheaper
#    runs while practising you can switch this to "claude-haiku-4-5".
MODEL = "claude-opus-4-8"
print("Using model:", MODEL)

## Step 1: three tools

In [ ]:
import ast
import operator

# A tool is described by a name, a plain description, and the inputs it needs.
# The model reads the description to decide when to use the tool.
calculator_tool = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression and return the result. Use this whenever the user asks for a calculation.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "An arithmetic expression, for example '12 * 37'.",
            }
        },
        "required": ["expression"],
    },
}

# The safe operations our calculator allows: plus, minus, times, divide, and
# negative numbers. Nothing else is permitted.
_SAFE_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _safe_eval(node):
    # Walk the parsed expression and only allow numbers and the safe operations
    # above. Anything else (names, function calls) raises an error, so no
    # arbitrary code can run. This is the safe alternative to eval().
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("unsupported expression")


def run_calculator(expression):
    # Beginners can treat this as a black box: give it an expression, get the
    # result. It parses the text and evaluates only safe arithmetic, so it
    # cannot run arbitrary code.
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_safe_eval(tree))
    except Exception:
        return "Error: I could only handle a simple arithmetic expression like '12 * 37'."

In [ ]:
# Two small tools. One looks up an approximate fact, one does arithmetic.
facts = {
    "geneva population": "200000 (approximate, for the workshop)",
    "zurich population": "430000 (approximate, for the workshop)",
    "bern population": "135000 (approximate, for the workshop)",
}

lookup_tool = {
    "name": "lookup_fact",
    "description": "Look up an approximate fact by name, for example 'geneva population'. Use this when you need a number you do not know.",
    "input_schema": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "The fact to look up, lowercase."}
        },
        "required": ["name"],
    },
}


def run_lookup(name):
    return facts.get(name.lower().strip(), "Error: I do not have that fact. Try another name.")

In [ ]:
# A third tool with a clearly different job: return a greeting in a language.
greetings = {
    "french": "Bonjour",
    "spanish": "Hola",
    "arabic": "Marhaba",
    "swahili": "Jambo",
}

greeting_tool = {
    "name": "greet_in_language",
    "description": "Return a greeting word in a given language. Use this when the user asks how to greet someone in another language.",
    "input_schema": {
        "type": "object",
        "properties": {
            "language": {"type": "string", "description": "The language, lowercase, for example 'french'."}
        },
        "required": ["language"],
    },
}


def run_greeting(language):
    return greetings.get(language.lower().strip(), "Error: I do not know that language yet.")

## Step 2: register the tools

Complete the tools list and the dispatcher so the agent can reach all three
tools. The instructor notebook has the answer.

In [ ]:
# TODO: include all three tools in this list.
tools = [calculator_tool, lookup_tool]  # add greeting_tool

def run_tool(name, tool_input):
    if name == "calculator":
        return run_calculator(tool_input["expression"])
    if name == "lookup_fact":
        return run_lookup(tool_input["name"])
    # TODO: add a branch for "greet_in_language" that calls run_greeting.
    return "Error: unknown tool."

SYSTEM = (
    "You are a helpful assistant. Use the tools when they fit the request, "
    "one step at a time, then give a short final answer."
)

## Step 3: the agent loop

This is the same loop you built in Notebook 2, ready to reuse.

In [ ]:
def run_agent(question, max_steps=5):
    messages = [{"role": "user", "content": question}]
    for step in range(max_steps):
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system=SYSTEM, tools=tools, messages=messages,
        )
        for block in response.content:
            if block.type == "text" and block.text.strip():
                print("Thought:", block.text.strip())
        if response.stop_reason != "tool_use":
            for block in response.content:
                if block.type == "text":
                    return block.text
            return ""
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print("Action:", block.name, block.input)
                observation = run_tool(block.name, block.input)
                print("Observation:", observation)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": observation})
        messages.append({"role": "user", "content": tool_results})
    return "Stopped after the maximum number of steps."

## Step 4: watch it choose

Each question below should make the agent reach for a different tool.

In [ ]:
print("=== Calculation ===")
print(run_agent("What is 45 plus 67? Use the calculator."))
print("\n=== Lookup ===")
print(run_agent("What is the approximate population of Bern?"))
print("\n=== Greeting ===")
print(run_agent("How do I say hello in Swahili?"))

## Your turn

Add a brand new tool of your own (for example a tool that reverses a piece of
text) and a question that uses it. You will need to: define the tool, add it to
the `tools` list, and add a branch in `run_tool`. This is also assignment A2.